In [1]:
import pytest
import ipytest
from gains import *

In [2]:
ipytest.autoconfig()

In [3]:
%%ipytest

@pytest.fixture
def test_env():
    """Setup a basic room, masters, and sensors."""
    # This assumes RoomBuilder, Room, and Element classes are defined in previous cells
    # or globally in the same cell.
    config = {
        'environment': {
            'dimensions': [5, 4, 3],
            'wall_resolution': (4, 4),
            'reflectivity': {'floor': 0.2, 'ceiling': 0.8, 'walls': 0.5}
        }
    }
    rb = RoomBuilder(config)
    room = Room(rb)
    
    # 1 Master (Tx) at ceiling, 1 Sensor (Rx) on floor
    masters = OpticalTxElements(r=[2.5, 2.0, 3.0], n=[0, 0, -1], p=1.0, m=1)
    sensors = OpticalRxElements(r=[2.5, 2.0, 0.0], n=[0, 0, 1], A=1e-4)
    
    return room, masters, sensors

def test_los_gain_magnitude(test_env):
    room, masters, sensors = test_env
    engine = Gains(room, sensors, masters)
    engine.los_channel_gains()
    
    assert engine.h_los.shape == (1, 1)
    assert engine.h_los[0, 0] > 0
    # At 3m distance, A=1e-4, m=1, gain should be ~ (1e-4 * 2)/(2 * pi * 3^2)
    expected = (1e-4 * 2) / (2 * np.pi * 3**2)
    assert engine.h_los[0, 0] == pytest.approx(expected, rel=1e-3)

def test_rf_gain_distance():
    # Test that RF path loss increases with distance
    tx = Elements(r=[0, 0, 0])
    rx1 = Elements(r=[0, 0, 1])
    rx2 = Elements(r=[0, 0, 10])
    
    pl1 = Gains.calc_h_rf(tx, rx1)
    pl2 = Gains.calc_h_rf(tx, rx2)
    assert pl2 > pl1

def test_diffuse_multi_bounce(test_env):
    room, masters, sensors = test_env
    engine = Gains(room, sensors, masters)
    
    # Check if h_ww was auto-initialized by Gains
    assert room.h_ww is not None
    
    engine.diffuse_channel_gains(bounces=2)
    assert engine.h_diff.shape == (1, 1)
    assert engine.h_diff[0, 0] > 0

...                                                                                          [100%]
3 passed in 0.10s
